# CV 配音(Colab T4 GPU 版)

本機顯示卡排不到時,把 C 段「凍結前」的配音丟到 Google Colab 免費 T4 跑。**可以中斷**:斷線、逾時、按停止都沒關係,重新連線後由上往下再執行一次,會從斷點接著做(續跑快取放在 Google Drive 的 `CV_colab/resume/<tag>/`)。全部用 Python(`pipeline/colab/colab_run.py`),不跑 sh。

使用者要做的三件事:

1. 本機:`python -X utf8 pipeline\colab\pack_for_colab.py <tag>` → `tmp\colab\colab_<tag>.zip`,上傳到 Google Drive 的 `CV_colab\`(或讓第 1 格跳出上傳鈕)。
2. Colab:開這本筆記本(已指定 **T4 GPU**;沒選到就 執行階段 → 變更執行階段類型 → T4 GPU),改第 1 格的 `TAG`,按「執行階段 → 全部執行」。第一次安裝約 10–15 分;做完 Drive 的 `CV_colab` 會有 `tts_<tag>.zip`。
   - 2、3 在背景跑(格子馬上結束);2b、3b 會一直轉到安裝/配音結束,按停止只停監看,背景照跑。
   - 同一個執行階段裝好過,第 2 格直接跳過。
3. 本機:下載 `tts_<tag>.zip`,`python -X utf8 pipeline\colab\pack_for_colab.py --import <下載的 zip>`,再 `python -X utf8 pipeline\make_version.py --tag <tag> --ver <ver> --tts-review --freeze`(全部命中快取、0 新合成 → 自動凍結)。

- 品管未過的句子:本機改寫 `script.txt` 後重新打包上傳、再「全部執行」;第 1b 格會接上 Drive 上的 `tts_<tag>.zip`(通過的沿用、沒通過的重錄)。
- 重錄上限維持 tts_cosy3 預設(換 seed 6 次、語速 6 次);要調高在第 1 格設 `CV_TTS_MAX_RETRY` / `CV_TTS_RATE_SEEDS`,上限 12。卡在同一批句子是文案問題,改寫文案,不要再錄。
- 免費版系統記憶體只有 12 GB:副辨識 Fun-ASR 上顯示卡(`TTS_ASR2_DEVICE=cuda`,第 1 格設)。
- 快取鍵只看參數不看路徑(`pipeline/tts_key.py` cmd_sig),Colab 做的音檔本機直接命中;品管規則與本機相同。配音一律「未聽審」:回本機照規則出聽審清單。

In [ ]:
# 1. 掛 Drive、取得壓縮檔(Drive 上沒有就跳出上傳鈕)、解壓、確認 GPU
TAG = 'quantum'   # 改成你的片 tag

import glob, json, os, shutil, subprocess, sys, threading, time, zipfile
from google.colab import drive, files
from IPython.display import clear_output
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/CV_colab'
os.makedirs(DRIVE, exist_ok=True)
ZIP = f'{DRIVE}/colab_{TAG}.zip'
if not os.path.exists(ZIP):
    up = files.upload()
    shutil.move(next(iter(up)), ZIP)
    print('已上傳到', ZIP)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, '沒有 GPU:執行階段 → 變更執行階段類型 → T4 GPU'
print('GPU:', gpu.stdout.strip())

CV = '/content/CV'
stamp, mark = f'{os.path.getsize(ZIP)}-{int(os.path.getmtime(ZIP))}', f'{CV}/.cv_zip'
if not (os.path.exists(mark) and open(mark).read() == stamp):   # 同一個執行階段重跑這格:壓縮檔沒換就不重解,不蓋掉做好的
    os.makedirs(CV, exist_ok=True)
    zipfile.ZipFile(ZIP).extractall(CV)
    open(mark, 'w').write(stamp)
    print('已解壓', ZIP)
os.chdir(CV)
assert os.path.exists('pipeline/colab/colab_run.py'), (f'{ZIP} 是舊版打包(沒有 pipeline/colab/colab_run.py):本機重新跑 '
                                                       'pack_for_colab.py,上傳覆蓋 Drive 上的同名檔,再重新整個執行')
os.environ['CLAUDE_VIDEO_ROOT'] = CV                   # 素材根 = 解壓目錄(paths.py)
os.environ['CV_COLAB_DRIVE'] = DRIVE
os.environ['TTS_RESUME_DIR'] = f'{DRIVE}/resume'      # 續跑快取放 Drive(<此目錄>/<tag>/),換新的執行階段也接得上
os.environ['TTS_RESUME_KEEP'] = '1'                    # 配音完先打包回 Drive 才刪(第 5 格)
os.environ['TTS_ASR2_DEVICE'] = 'cuda'                 # 副辨識 Fun-ASR 上顯示卡:免費版 12 GB 記憶體放不下它在 CPU 的 3.6 GB
os.environ['TTS_TS_DEVICE'] = 'cuda'                   # 字詞時間戳 faster-whisper small 上顯示卡
# os.environ['CV_TTS_MAX_RETRY'] = '12'                # 換 seed 重錄上限(預設 6,上限 12);卡在同一批句子是文案問題,改寫文案
# os.environ['CV_TTS_RATE_SEEDS'] = '12'               # 語速重錄上限(預設 6,上限 12)
sys.path.insert(0, f'{CV}/pipeline/colab')
import colab_run as CR
LOG = f'{DRIVE}/tts_{TAG}.log'                        # 配音紀錄放 Drive:斷線後也看得到
SETUP_LOG, SETUP_PID, SETUP_RC = '/content/setup.log', '/content/setup.pid', '/content/setup_cv.rc'
PIDF, RCF = f'{CV}/build/{TAG}/.cv_tts.pid', f'{CV}/build/{TAG}/.cv_tts.rc'
KEYS = ('[resume]', '[colab]', 'pass1', 'retry', 'rate ', '[cosy3]', 'TTS', '警告', 'Error', 'error')


def alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def tail(path, n=8, keys=None):
    if not os.path.exists(path):
        return '(尚無紀錄)'
    ls = open(path, encoding='utf-8', errors='replace').read().replace('\r', '\n').splitlines()
    if keys:
        ls = [l for l in ls if any(k in l for k in keys)]
    return '\n'.join(l[:200] for l in ls[-n:])


def background(cmd, log, pidf, rcf, **kw):   # 背景行程:pid 檔現在寫、rc 檔結束才寫(2b/3b 靠 rc 檔判斷結束)
    if os.path.exists(rcf):
        os.remove(rcf)
    p = subprocess.Popen(cmd, cwd=CV, stdout=log, stderr=subprocess.STDOUT, start_new_session=True, **kw)
    open(pidf, 'w').write(str(p.pid))

    def wait():
        rc = p.wait()
        open(rcf, 'w').write(str(rc))
    threading.Thread(target=wait, daemon=True).start()


CR.status(TAG)

In [ ]:
# 1b. 接上 Drive 上已做好的配音(CV_colab/tts_<tag>.zip,第 5 格每次做完都會更新它):通過的直接沿用,沒通過的重錄
#     沒有這個檔(第一次配這支片)就跳過;同一個執行階段已接過、檔案沒換就不重接(不蓋掉這次新做的)
TTS_ZIP = f'{DRIVE}/tts_{TAG}.zip'
tts_mark = f'{CV}/.cv_tts_zip'
if not os.path.exists(TTS_ZIP):
    print('Drive 上沒有', TTS_ZIP, ',全部從頭配')
else:
    st = f'{os.path.getsize(TTS_ZIP)}-{int(os.path.getmtime(TTS_ZIP))}'
    if os.path.exists(tts_mark) and open(tts_mark).read() == st:
        print('這個執行階段已接過', TTS_ZIP)
    else:
        n, redo = CR.apply_tts_zip(TAG, TTS_ZIP, CV)   # 解出 build/<tag>/**;沒通過品管的刪 .key -> 這次重錄,通過的沿用
        open(tts_mark, 'w').write(st)
        print(f'已接上 {TTS_ZIP}:{n} 個檔;沒通過、這次重錄 {len(redo)} 句:{redo}')

In [ ]:
# 2. 背景安裝 4 個環境與模型;這個執行階段裝好過、或正在裝,就跳過;進度看第 2b 格
if os.path.exists(SETUP_RC) and open(SETUP_RC).read().strip() == '0':
    print('這個執行階段已經裝好,跳過')
elif os.path.exists(SETUP_PID) and alive(int(open(SETUP_PID).read())) and not os.path.exists(SETUP_RC):
    print('安裝正在進行中,不重開;進度看第 2b 格')
else:   # 沒裝過或上次失敗:安裝(已裝好的環境與模型會跳過,只補沒完成的)
    background([sys.executable, '-X', 'utf8', 'pipeline/colab/colab_run.py', 'setup'], open(SETUP_LOG, 'w'), SETUP_PID, SETUP_RC)
    print('安裝已在背景啟動')

In [ ]:
# 2b. 看安裝進度:一直轉到安裝結束(按停止只停監看,背景安裝不受影響)
def show_setup():
    print('== 安裝', time.strftime('%H:%M:%S'))
    for f in sorted(glob.glob(f'{CV}/tmp/setup_logs/*')):
        print(f'  {os.path.basename(f):26s} {tail(f, 1).strip()[-110:] or "…"}')
    print(subprocess.run('du -sh /content/CV/tts_env_* /content/CV/tts_models 2>/dev/null', shell=True,
                         capture_output=True, text=True).stdout)
    print(tail(SETUP_LOG, 4))

while not os.path.exists(SETUP_RC):
    clear_output(wait=True)
    show_setup()
    time.sleep(15)
clear_output(wait=True)
print(tail(SETUP_LOG, 20))
rc = open(SETUP_RC).read().strip()
assert rc == '0', f'安裝失敗 rc={rc}:看上面與 {CV}/tmp/setup_logs/;修好後重跑第 2 格'
print('安裝完成,接著跑第 3 格啟動配音')

In [ ]:
# 3. 背景配音(安裝完成後才跑;colab_run.tts_cmd/tts_env -> build.py --tts-only --tts-engine cosy3);中斷後重跑這格會從斷點接著做
assert os.path.exists(SETUP_RC) and open(SETUP_RC).read().strip() == '0', '安裝還沒完成或失敗(看第 2b 格)'
os.makedirs(f'build/{TAG}', exist_ok=True)
if os.path.exists(PIDF) and alive(int(open(PIDF).read())) and not os.path.exists(RCF):
    print('配音已在跑,接著看第 3b 格')
else:
    with open(LOG, 'a') as f:
        f.write(f'\n===== {time.strftime("%Y-%m-%d %H:%M:%S")} 開始 =====\n')
    background(CR.tts_cmd(TAG), open(LOG, 'a'), PIDF, RCF, env=CR.tts_env())
    print('配音已在背景啟動,紀錄:', LOG)

In [ ]:
# 3b. 看配音進度:一直轉到配音結束(按停止只停監看,背景配音不受影響)
while not os.path.exists(RCF):
    clear_output(wait=True)
    print('== 配音中', time.strftime('%H:%M:%S'))
    print(tail(LOG, 10, KEYS))
    time.sleep(15)
clear_output(wait=True)
print(tail(LOG, 15, KEYS))
print('\n== 配音結束 rc=' + open(RCF).read().strip(), '(0 = 成功,接著跑第 5 格打包;不是 0 就看紀錄,修好後重跑第 3 格會從斷點接著做)')

In [ ]:
# 4.(選用)看做到哪裡(已完成單位、_jobs.json 剩幾個、續跑快取)、記憶體與顯存;斷線重連後先跑第 1 格再跑這格
CR.status(TAG)
print(subprocess.run('free -m | head -2; nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout)

In [ ]:
# 5. 配音成功後:打包回 Drive CV_colab/tts_<tag>.zip(先寫 .part 再改名)-> 打包成功才刪續跑快取與工作檔 -> 印品管
assert os.path.exists(RCF) and open(RCF).read().strip() == '0', f'配音未成功,看 {LOG}'
print(subprocess.run(['grep', '-a', '-E', 'TTS 完成|未通過|qc 沿用|警告|續跑快取', LOG], capture_output=True, text=True).stdout[-1500:])
out = CR.pack(TAG)
CR.clean(TAG, resume=True)
q = json.load(open(f'build/{TAG}/tts_qc.json', encoding='utf-8'))
print(f"品管 {q['passed']}/{q['checked']} 通過;未通過:{[x['idx'] for x in q['failed']] or '無'}(配音未聽審,回本機出聽審清單)")
print('下載', out, '-> 本機 pack_for_colab.py --import')